# 03 — Analysis and Validation

Build an `EModelAnalysisAndValidationScanConfig`, expand it via `GridScanGenerationTask`,
and run the `EModelAnalysisAndValidationTask` for each coordinate. The task seeds its
working directory from the optimisation output, merges the analysis-related
`pipeline_settings` from the blocks into `recipes.json`, then runs
`pipeline.store_optimisation_results()` → `pipeline.validation()` →
`pipeline.plot(only_validated=...)`.

**Reads from:** `obi-output/02_emodel_optimization/grid_scan/0/`.

**Writes to:** `obi-output/03_analysis_and_validation/grid_scan/0/` — the working
directory plus the new `final.json` and the `figures/L5PC/` plots (traces, scores,
distributions, currentscape if enabled).


## Imports

In [1]:
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.emodel_optimization._03_analysis_and_validation.blocks import (
    AnalysisInitialize,
    AnalysisSettings,
    CurrentscapeConfig,
)


## Build the scan config

In [2]:
previous_stage = (
    Path.cwd() / "../../../obi-output/02_emodel_optimization/grid_scan/0"
).resolve()
assert previous_stage.exists(), (
    f"{previous_stage} not found — run 02_emodel_optimization.ipynb first."
)

scan_config = obi.EModelAnalysisAndValidationScanConfig(
    initialize=AnalysisInitialize(
        previous_stage_output_path=previous_stage,
        emodel="L5PC",
        species="rat",
        brain_region="SSCX",
    ),
    analysis_settings=AnalysisSettings(
        validation_protocols=("sAHP_220",),
        plot_currentscape=True,
        save_recordings=False,
        only_validated_plots=False,
    ),
    currentscape_config=CurrentscapeConfig(figure_title="L5PC"),
)
print(scan_config.analysis_settings.to_dict(scan_config.currentscape_config))


{'validation_protocols': ['sAHP_220'], 'plot_currentscape': True, 'save_recordings': False, 'currentscape_config': {'title': 'L5PC'}}


## Run the grid scan

In [3]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/03_analysis_and_validation/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)


[2026-05-05 14:33:39,322] INFO: Seeded /Users/james/Documents/obi/code/obi-main/obi-output/02_emodel_optimization/grid_scan/0/config -> /Users/james/Documents/obi/code/obi-main/obi-output/03_analysis_and_validation/grid_scan/0/config
[2026-05-05 14:33:39,323] INFO: Seeded /Users/james/Documents/obi/code/obi-main/obi-output/02_emodel_optimization/grid_scan/0/morphologies -> /Users/james/Documents/obi/code/obi-main/obi-output/03_analysis_and_validation/grid_scan/0/morphologies
[2026-05-05 14:33:39,326] INFO: Seeded /Users/james/Documents/obi/code/obi-main/obi-output/02_emodel_optimization/grid_scan/0/mechanisms -> /Users/james/Documents/obi/code/obi-main/obi-output/03_analysis_and_validation/grid_scan/0/mechanisms
[2026-05-05 14:33:39,400] INFO: Seeded /Users/james/Documents/obi/code/obi-main/obi-output/02_emodel_optimization/grid_scan/0/ephys_data -> /Users/james/Documents/obi/code/obi-main/obi-output/03_analysis_and_validation/grid_scan/0/ephys_data
[2026-05-05 14:33:39,402] INFO: Seed

--No graphics will be displayed.


[2026-05-05 14:33:39,689] WARNING: The iteration is None. It is strongly advised to use an iteration tag in the future.
[2026-05-05 14:33:39,689] WARNING: Entry L5PC__1 was already in the final.json and will be overwritten
[2026-05-05 14:33:39,873] INFO: In compute_responses, 1 emodels found to evaluate.
[2026-05-05 14:34:31,890] INFO: In validate, 1 emodels found to validate.
[2026-05-05 14:34:33,583] WARNING: The iteration is None. It is strongly advised to use an iteration tag in the future.
[2026-05-05 14:34:33,584] WARNING: Entry L5PC__1 was already in the final.json and will be overwritten
[2026-05-05 14:34:33,868] INFO: maxp pruned
[2026-05-05 14:34:33,873] INFO: cmap pruned
[2026-05-05 14:34:33,874] INFO: kern dropped
[2026-05-05 14:34:33,874] INFO: post pruned
[2026-05-05 14:34:33,875] INFO: FFTM dropped
[2026-05-05 14:34:33,876] INFO: GPOS pruned
[2026-05-05 14:34:33,878] INFO: GSUB pruned
[2026-05-05 14:34:33,880] INFO: glyf pruned
[2026-05-05 14:34:33,881] INFO: Added gid0 

[PosixPath('/Users/james/Documents/obi/code/obi-main/obi-output/03_analysis_and_validation/grid_scan/0')]

<Figure size 640x480 with 0 Axes>

## Inspect the validated models

In [4]:
import json

coord_root = Path(grid_scan.single_configs[0].coordinate_output_root).resolve()
print("Coordinate output:", coord_root)

final_path = coord_root / "final.json"
if final_path.exists():
    final = json.loads(final_path.read_text())
    print(f"final.json — {len(final)} model(s)")
    for emodel_name, payload in final.items():
        print(" ", emodel_name, "validated:", payload.get("validated", payload.get("passed_validation")))
else:
    print("final.json not produced yet")

figures = sorted((coord_root / "figures").rglob("*.pdf")) + sorted(
    (coord_root / "figures").rglob("*.png")
)
print(f"\nFigures: {len(figures)}")
for p in figures[:8]:
    print(" ", p.relative_to(coord_root))


Coordinate output: /Users/james/Documents/obi/code/obi-main/obi-output/03_analysis_and_validation/grid_scan/0
final.json — 1 model(s)
  L5PC__1 validated: False

Figures: 62
  figures/L5PC/currentscape/all/emodel=L5PC__species=rat__brain_region=SSCX__seed=1__currentscape.APWaveform_280.0.soma.pdf
  figures/L5PC/currentscape/all/emodel=L5PC__species=rat__brain_region=SSCX__seed=1__currentscape.IDrest_150.0.soma.pdf
  figures/L5PC/currentscape/all/emodel=L5PC__species=rat__brain_region=SSCX__seed=1__currentscape.IDrest_250.0.soma.pdf
  figures/L5PC/currentscape/all/emodel=L5PC__species=rat__brain_region=SSCX__seed=1__currentscape.IV_-100.0.soma.pdf
  figures/L5PC/currentscape/all/emodel=L5PC__species=rat__brain_region=SSCX__seed=1__currentscape.sAHP_220.0.soma.pdf
  figures/L5PC/distributions/all/emodel=L5PC__species=rat__brain_region=SSCX__parameters_distribution.pdf
  figures/L5PC/efeatures_extraction/C060109A1-SR-C1/C060109A1-SR-C1_APWaveform_AHP_depth_amp.pdf
  figures/L5PC/efeatures